In [1]:
%load_ext autoreload
%autoreload 2

import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("optuna").setLevel(logging.WARNING)

import numpy as np
import pandas as pd
import optuna
import MetaTrader5 as mt5
from sklearn.ensemble import IsolationForest

from trading_engine import (
    get_mt5_data, split_data, apply_trading_costs,
    apply_sl_tp_and_costs, calculate_ai_metrics, generate_tearsheet,
)
from instruments import get_market_watch_symbols  # optional symbol scanner

# quant_toolbox also exposes: fit_strategy, run_backtest, stress_test, walk_forward,
# get_swap_pct, get_ref_atr, add_atr, STOP_MODES, ...
from quant_toolbox import *

optuna.logging.set_verbosity(optuna.logging.WARNING)

if not mt5.initialize():
    raise RuntimeError(f"MT5 initialize() failed: {mt5.last_error()}")
print("MT5 connected:", mt5.terminal_info().name)


MT5 connected: MetaTrader 5 EXNESS


In [ ]:
# ----------------------------- CONFIG -----------------------------
SYMBOLS      = get_market_watch_symbols()     # or ["EURUSD", "GBPUSD", "XAUUSD", "USDJPY"]
TIMEFRAME    = mt5.TIMEFRAME_H1
N_CANDLES    = 20000
TRAIN_PCT    = 0.70
VARIANTS     = ["daily", "3std", "outlier", "bb"]   # one Optuna study PER variant
N_TRIALS_PER_VARIANT = 60
MIN_ACTIVE_DAYS = 30                                # anti-overfit floor (counts active days)
MIN_IS_FITNESS  = 0.0                               # skip OOS if best in-sample fitness <= this
GRADED_PENALTY  = True                              # graded scores for failed trials (False = flat -9999)
SWAP_DEFAULT    = 0.0002                            # fallback swap, fraction of price per night (0.02%)
IF_LOOKBACK  = 1000
IF_REFIT     = 500
SEED         = 42

# ------------------------ STRATEGY: signals + search space ------------------------
def _hold_state(long_ev, short_ev, exit_ev=None):
    n = len(long_ev); out = np.zeros(n); s = 0
    for i in range(n):
        if long_ev[i]:    s = 1
        elif short_ev[i]: s = -1
        elif exit_ev is not None and exit_ev[i]: s = 0
        out[i] = s
    return out

def _rising_edge(mask):
    return mask & ~np.r_[False, mask[:-1]]

def causal_outlier_flags(dev, lookback, refit_every, contamination, seed):
    """Walk-forward IsolationForest: each block is scored by a model trained only on prior bars."""
    n = len(dev); flags = np.zeros(n, dtype=bool)
    for i in range(lookback, n, refit_every):
        tr = dev[i - lookback:i]; tr = tr[~np.isnan(tr)]
        if len(tr) < 100:
            continue
        model = IsolationForest(n_estimators=50, contamination=contamination,
                                random_state=seed, n_jobs=1).fit(tr.reshape(-1, 1))
        blk = dev[i:i + refit_every]; ok = ~np.isnan(blk)
        if ok.any():
            flags[np.where(ok)[0] + i] = (model.predict(blk[ok].reshape(-1, 1)) == -1)
    return flags

def compute_strategy_signals(df, variant, k=20, break_event=0.01, z_thresh=3.0,
                             contamination=0.05, asset_ma=1, bb_ma=100, seed=SEED):
    """Target positions (+1 / -1 / 0) decided at close[t]; the engine trades them at open[t+1]."""
    close = df["close"]
    if variant in ("daily", "3std", "outlier"):
        sma = close.rolling(int(k)).mean()
        dev = close / sma - 1.0
        if variant == "daily":
            ev = (dev.abs() > break_event).values
        elif variant == "3std":
            sd = close.rolling(int(k)).std()
            ev = (((close - sma) / sd).abs() > z_thresh).values
        else:
            ev = causal_outlier_flags(dev.values, IF_LOOKBACK, IF_REFIT, contamination, seed)
        d = dev.values
        pos = _hold_state(ev & (d < 0), ev & (d > 0), exit_ev=None)   # paper: no mean-cross exit
    elif variant == "bb":
        fast = close.rolling(int(asset_ma)).mean() if asset_ma > 1 else close
        ma   = close.rolling(int(bb_ma)).mean()
        sd   = close.rolling(int(bb_ma)).std()
        upper, lower = ma + 2 * sd, ma - 2 * sd
        long_ev  = _rising_edge((fast < lower).values)
        short_ev = _rising_edge((fast > upper).values)
        side = np.sign((fast - ma).values)
        prev = np.r_[np.nan, side[:-1]]
        exit_ev = (side != prev) & ~np.isnan(prev) & (prev != 0) & ~np.isnan(side)
        pos = _hold_state(long_ev, short_ev, exit_ev=exit_ev)         # paper: exit at BB mean
    else:
        raise ValueError(variant)
    return pd.Series(pos, index=df.index, name="position")

def suggest_params(trial, variants):
    """Strategy params only. Stop params come from suggest_stop_params()."""
    p = {"variant": trial.suggest_categorical("variant", variants)}
    v = p["variant"]
    if v == "daily":
        p["k_daily"]     = trial.suggest_int("k_daily", 2, 200)
        p["break_event"] = trial.suggest_float("break_event", 0.0005, 0.03, log=True)
    elif v == "3std":
        p["k_3std"]      = trial.suggest_int("k_3std", 5, 100)
        p["z_thresh"]    = trial.suggest_float("z_thresh", 2.0, 3.5, step=0.25)
    elif v == "outlier":
        p["k_out"]         = trial.suggest_int("k_out", 50, 500, step=5)
        p["contamination"] = trial.suggest_float("contamination", 0.01, 0.10, step=0.005)
    else:
        p["asset_ma"] = trial.suggest_int("asset_ma", 1, 5)
        p["bb_ma"]    = trial.suggest_int("bb_ma", 10, 900, step=5)
    return p

def map_params(p):
    """Flat Optuna params -> kwargs for compute_strategy_signals."""
    v = p["variant"]; kw = {"variant": v}
    if v == "daily":     kw.update(k=p["k_daily"], break_event=p["break_event"])
    elif v == "3std":    kw.update(k=p["k_3std"], z_thresh=p["z_thresh"])
    elif v == "outlier": kw.update(k=p["k_out"], contamination=p["contamination"])
    else:                kw.update(asset_ma=p["asset_ma"], bb_ma=p["bb_ma"])
    return kw

# ----------------------- SCREEN: optimise IS -> blind OOS -> cost stress -----------------------
portfolio_results, stress_tables = [], {}

for sym in SYMBOLS:
    print(f"\n=== {sym} ===")
    if not mt5.symbol_select(sym, True):
        print(f"  skip: cannot select {sym}"); continue
    info = mt5.symbol_info(sym)
    if info is None:
        print(f"  skip: no symbol_info for {sym}"); continue
    point_size, spread_pts = info.point, info.spread

    df = get_mt5_data(sym, TIMEFRAME, N_CANDLES)
    if df is None or len(df) < 3000:
        print("  skip: insufficient data"); continue

    train_df, test_df = split_data(df, train_pct=TRAIN_PCT)

    fit = fit_strategy(train_df, point_size, spread_pts, compute_strategy_signals, suggest_params,
                       map_params, VARIANTS, STOP_MODES, n_trials=N_TRIALS_PER_VARIANT, seed=SEED,
                       min_active_days=MIN_ACTIVE_DAYS, graded=GRADED_PENALTY)
    if fit is None or fit["fitness"] <= MIN_IS_FITNESS:
        print(f"  no viable in-sample strategy, skipping OOS"); continue
    best, stop, ref_atr = fit["params"], fit["stop"], fit["ref_atr"]
    print(f"  WINNER {best['variant']} | stop {stop} | IS fitness {fit['fitness']:.4f}")

    # Blind OOS test, run ONCE (signals computed on full history so windows are warm)
    pos_oos = compute_strategy_signals(df, **map_params(best)).loc[test_df.index]
    oos_net = run_backtest(test_df, pos_oos, point_size, spread_pts, ref_atr, stop)
    oos = calculate_ai_metrics(oos_net, mc_simulations=500)
    if oos is None:
        print("  OOS metrics unavailable"); continue

    # Cost stress on the OOS slice: same positions + stop, harsher costs
    swap = get_swap_pct(info, float(train_df["close"].median()), default=SWAP_DEFAULT)
    stress_tbl, stress = stress_test(test_df, pos_oos, point_size, spread_pts, ref_atr, stop,
                                     swap_pct_per_night=swap)
    stress_tables[sym] = stress_tbl

    bh = calculate_ai_metrics(test_df["close"].pct_change().dropna(), mc_simulations=100)
    generate_tearsheet(oos_net, title=f"{sym} {best['variant']} {stop['mode']} OOS")

    portfolio_results.append({
        "symbol": sym, "variant": best["variant"],
        "stop_mode": stop["mode"], "stop_atr_mult": stop["mult"],
        "is_fitness": fit["fitness"],
        "oos_cagr": oos["cagr"], "oos_sharpe": oos["sharpe"],
        "oos_sortino": oos["sortino"], "oos_max_dd": oos["max_drawdown"],
        "oos_calmar": oos["calmar"], "oos_win_rate": oos["win_rate"],
        "oos_profit_factor": oos["profit_factor"], "oos_active_days": oos["n_trades"],
        "oos_mc_worst5_dd": oos["mc_worst_5%_drawdown"],
        **stress, "swap_pct_night": swap,
        "bh_sharpe": bh["sharpe"] if bh else np.nan,
        "params": best,
    })

if portfolio_results:
    leaderboard = (pd.DataFrame(portfolio_results)
                     .sort_values("oos_sharpe", ascending=False).reset_index(drop=True))
    display(leaderboard)
else:
    print("No symbol produced a viable in-sample strategy.")


✅ Found 47 instruments in MT5 Market Watch.

=== BTCUSD ===
Data Split: 14000 Training Candles | 6000 Testing Candles
  IS fitness by group: {'daily': -0.0519, '3std': -0.0391, 'outlier': -0.0507, 'bb': -0.0685}
  no viable in-sample strategy, skipping OOS

=== USOIL ===
Data Split: 14000 Training Candles | 6000 Testing Candles
  IS fitness by group: {'daily': -0.2664, '3std': -0.0612, 'outlier': -0.0111, 'bb': -0.0425}
  no viable in-sample strategy, skipping OOS

=== XNGUSD ===
Data Split: 14000 Training Candles | 6000 Testing Candles
  IS fitness by group: {'daily': -0.485, '3std': -0.0998, 'outlier': -0.0872, 'bb': -0.1083}
  no viable in-sample strategy, skipping OOS

=== AUDCAD ===
Data Split: 14000 Training Candles | 6000 Testing Candles
  IS fitness by group: {'daily': -0.0259, '3std': -0.0248, 'outlier': -0.0118, 'bb': -0.0031}
  no viable in-sample strategy, skipping OOS

=== AUDCHF ===
Data Split: 14000 Training Candles | 6000 Testing Candles
  IS fitness by group: {'daily':

In [ ]:
# ------------- WALK-FORWARD on the shortlist (re-optimises every fold: slow, run on candidates only) -------------
from functools import partial

WF_FOLDS       = 5
WF_INIT_TRAIN  = 0.5        # first 50% = first training set; the rest is split into WF_FOLDS test chunks
WF_ANCHORED    = True       # True = expanding train window, False = rolling window of the initial size
WF_TRIALS      = 30         # trials per variant per fold (5 folds x 4 variants x 30 = 600 backtests / symbol)
WF_MIN_POS_FOLDS = 0.8     # share of folds that must be profitable for wf_pass (0.8 = 4 of 5; stricter = fewer false positives)

# Candidates that were profitable OOS AND survived every cost scenario. Or set a list by hand.
SHORTLIST = (leaderboard.loc[leaderboard["survives_stress"] & (leaderboard["oos_sharpe"] > 0), "symbol"]
                        .head(5).tolist() if portfolio_results else [])
# SHORTLIST = ["XNGUSD", "EURJPY"]

wf_rows, wf_results = [], {}
for sym in SHORTLIST:
    print(f"\n=== WALK-FORWARD {sym} ===")
    mt5.symbol_select(sym, True)
    info = mt5.symbol_info(sym)
    point_size, spread_pts = info.point, info.spread
    df = get_mt5_data(sym, TIMEFRAME, N_CANDLES)

    swap = get_swap_pct(info, float(df["close"].median()), default=SWAP_DEFAULT)
    fit_fn = partial(fit_strategy, point_size=point_size, spread_pts=spread_pts,
                     signal_fn=compute_strategy_signals, space_fn=suggest_params, map_fn=map_params,
                     groups=VARIANTS, stop_modes=STOP_MODES, n_trials=WF_TRIALS, seed=SEED,
                     min_active_days=MIN_ACTIVE_DAYS, graded=GRADED_PENALTY, verbose=False)
    wf = walk_forward(df, fit_fn, compute_strategy_signals, map_params, point_size, spread_pts,
                      n_folds=WF_FOLDS, init_train_frac=WF_INIT_TRAIN, anchored=WF_ANCHORED,
                      min_is_fitness=MIN_IS_FITNESS, min_pos_fold_share=WF_MIN_POS_FOLDS,
                      engine_kw={"swap_pct_per_night": swap})   # walk-forward already includes swap
    wf_results[sym] = wf
    display(wf["folds"].drop(columns="params"))
    wf_rows.append({"symbol": sym, **wf["summary"]})
    generate_tearsheet(wf["returns"], title=f"{sym} walk-forward stitched OOS")

if wf_rows:
    wf_summary = pd.DataFrame(wf_rows).sort_values("wf_sharpe", ascending=False).reset_index(drop=True)
    display(wf_summary)
else:
    print("Empty shortlist - nothing survived the OOS + cost-stress screen.")


In [ ]:
mt5.shutdown()
print("MT5 connection closed.")
